# remote

> Hosted models through [fastllm](https://github.com/AnswerDotAI/fastllm). The same `Chat` API, with API keys in the environment (`ANTHROPIC_API_KEY`, `OPENAI_API_KEY`, `GEMINI_API_KEY`, ...).

Portable `hist` lets you start local and hand off to Claude/GPT/Gemini without reformatting.


In [ ]:
#| default_exp remote

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
from base64 import b64decode
from fastllm.acomplete import acomplete
from fastllm.types import Completion
from aidialog.msg_parts import (Msg, Part, PartType, Text, Thinking, ToolUse, ToolResult,
                                InputImage, InputAudio, data_url)
from fastcore.all import store_attr, ifnone, listify
from rishi import core
from rishi.core import *

In [ ]:
#| export
def _client_httpx():
    "The httpx the OpenAPI client itself imported: fastspec is on httpx2 in some installs, httpx in others."
    import fastspec.oapi as o
    return getattr(o, 'httpx2', None) or getattr(o, 'httpx', None)

def _fix_client_timeout():
    "fastllm builds `mk_client`'s default timeout with its own httpx, which need not be the client's."
    import fastllm.acomplete as fa
    mod = _client_httpx()
    if mod is None: return
    f = getattr(fa.mk_client, '__wrapped__', fa.mk_client)
    def swap(x):
        if isinstance(x, mod.Timeout) or not all(hasattr(x, k) for k in ('connect','read','write','pool')): return x
        return mod.Timeout(connect=x.connect, read=x.read, write=x.write, pool=x.pool)
    f.__defaults__ = tuple(swap(x) for x in (f.__defaults__ or ()))
    cache = getattr(fa.mk_client, 'cache', None)
    if cache is not None:
        try: cache.clear()
        except Exception: pass

_fix_client_timeout()

In [ ]:
#| hide
# Silent until a request is made: the client hands the object to `anyio.fail_after`, which adds it
# to a float, and every hosted turn died on a TypeError naming neither library.
import inspect, fastllm.acomplete as fa
from fastcore.test import test_eq
_d = inspect.signature(getattr(fa.mk_client, '__wrapped__', fa.mk_client)).parameters['timeout'].default
test_eq(isinstance(_d, _client_httpx().Timeout), True)
test_eq((_d.connect, _d.read, _d.write, _d.pool), (30, 300, 30, 10))

In [ ]:
#| export
_all_ = ['UsageStats', 'ChatCallback', 'run_cbs', 'resp_text', 'thought', 'Resp', 'StreamFormatter',
         'display_stream', 'mk_tr_details', 'truncated', 'hitl_policy', 'extract_fence', 'mk_toolspec',
         'ToolCall', 'mk_tool_res_msg']

In [ ]:
from fastcore.test import test_eq, test_fail, test_close
from fastllm.types import Usage

## Messages

fastllm's canonical message is a `Msg(role, content=[Part, ...])` with typed parts: `text`,
`thinking`, `tool_use`, `tool_result`, `input_image` and `input_audio`. A media part carries a data
URL in `part.text`. rishi's canonical history is OpenAI-shaped dicts. Neither form is richer than the
other in practice, so this is a straight two-way mapping.

It is also where media survives a backend hop or does not. Images and audio are carried through as
data URLs rather than collapsed to a placeholder. Thinking round-trips too. rishi keeps it in
`channels.thought`, fastllm keeps it as a `thinking` part, and each converts to the other.

In [ ]:
#| export
def _parts(content):
    "Canonical rishi content (str or list of parts) -> aidialog `Part`s."
    if content is None: return []
    if isinstance(content, str): return [Text(text=content)] if content else []   # no empty part on a tool-call turn
    out = []
    for p in content:
        if not isinstance(p, dict): continue
        t = p.get('type')
        if   t == 'text' and (text := p.get('text', '')): out.append(Text(text=text))
        elif t in ('image_url','input_audio') and (media := to_media_part(p)): out.append(media)
    return out

def to_msg(m):
    "One canonical rishi history dict -> a fastllm `Msg`."
    role = m.get('role', 'user')
    if role == 'tool':
        return Msg(role='tool', content=[ToolResult(id=m.get('tool_call_id'), name=m.get('name', ''),
                                                   text=str(m.get('content', '')))])
    parts = []
    if role == 'assistant' and (th := (m.get('channels') or {}).get('thought')):
        parts.append(Thinking(text=th))
    parts += _parts(m.get('content'))
    for tc in (m.get('tool_calls') or []):
        fn = tc.get('function') or {}
        parts.append(ToolUse(id=tc.get('id'), name=fn.get('name', ''), arguments=fn.get('arguments') or {},
                             server=bool(tc.get('server'))))
    return Msg(role=role, content=parts)

def to_hist(m):
    "A fastllm `Msg` -> canonical rishi history dicts (a tool `Msg` can hold several results)."
    if m.role == 'tool':
        return [{'role': 'tool', 'tool_call_id': p.id, 'name': p.name or '', 'content': str(p.text)}
                for p in m.content if p.type == PartType.tool_result]
    text = ''.join(p.text or '' for p in m.content if p.type in (PartType.text, PartType.refusal))
    th   = ''.join(p.text or '' for p in m.content if p.type == PartType.thinking)
    media = [p for p in m.content if p.type in (PartType.input_image, PartType.input_audio)]
    out = {'role': m.role, 'content': text}
    if media:
        parts = ([{'type': 'text', 'text': text}] if text else []) + [_media_part(p) for p in media]
        out['content'] = parts
    if th: out['channels'] = {'thought': th}
    tcs = [ToolCall(name=p.name or '', arguments=p.arguments or {}, id=p.id, server=bool(p.server))
           for p in m.content if p.type == PartType.tool_use]
    if tcs: out['tool_calls'] = tcs
    return [out]

def _media_part(p):
    "An aidialog media `Part` -> an OpenAI-style content part."
    if p.type == PartType.input_image: return {'type': 'image_url', 'image_url': {'url': p.text}}
    mime, b64 = data_url(p.text) or ('audio/wav', '')
    return {'type': 'input_audio', 'input_audio': {'data': b64, 'format': mime.split('/')[-1]}}

In [ ]:
# a round trip through fastllm's Msg keeps text, thinking, tool calls and tool results intact
h = [{'role': 'user', 'content': 'what is 2+2?'},
     {'role': 'assistant', 'content': 'let me add', 'channels': {'thought': 'hmm'},
      'tool_calls': [ToolCall('add', {'a': 2, 'b': 2}, id='c1')]},
     {'role': 'tool', 'tool_call_id': 'c1', 'name': 'add', 'content': '4'},
     {'role': 'assistant', 'content': 'it is 4'}]
msgs = [to_msg(m) for m in h]
test_eq([m.role for m in msgs], ['user', 'assistant', 'tool', 'assistant'])
back = [x for m in msgs for x in to_hist(m)]
test_eq(back[0], {'role': 'user', 'content': 'what is 2+2?'})
test_eq(back[1]['channels'], {'thought': 'hmm'})
test_eq(back[1]['tool_calls'][0].name, 'add')
test_eq(back[1]['tool_calls'][0].arguments, {'a': 2, 'b': 2})
test_eq(back[2], {'role': 'tool', 'tool_call_id': 'c1', 'name': 'add', 'content': '4'})
test_eq(back[3], {'role': 'assistant', 'content': 'it is 4'})

# reverse conversion groups text first, then images and audio
mixed = [{'type': 'text', 'text': 'before'},
         {'type': 'image_url', 'image_url': {'url': 'data:image/png;base64,iVBORw0KGgo='}},
         {'type': 'image_url', 'image_url': {}},
         {'type': 'unknown'},
         {'type': 'text', 'text': 'between'},
         {'type': 'input_audio', 'input_audio': {'data': 'UklGRg==', 'format': 'wav'}},
         {'type': 'input_audio', 'input_audio': {}},
         {'type': 'text', 'text': 'after'}]
m = to_msg({'role': 'user', 'content': mixed})
test_eq([p.type for p in m.content], ['text', 'input_image', 'input_image', 'text', 'input_audio', 'input_audio', 'text'])
rt = to_hist(m)[0]
test_eq([p['type'] for p in rt['content']], ['text', 'image_url', 'image_url', 'input_audio', 'input_audio'])
test_eq(rt['content'][0]['text'], 'beforebetweenafter')
test_eq(rt['content'][1]['image_url']['url'], m.content[1].text)
test_eq(rt['content'][2]['image_url']['url'], m.content[2].text)
test_eq(rt['content'][3]['input_audio']['format'], 'wav')

# a server-side tool call keeps its flag across the hop
sm = to_msg({'role': 'assistant', 'content': '', 'tool_calls': [ToolCall('web_search', {}, id='s1', server=True)]})
assert to_hist(sm)[0]['tool_calls'][0].server

## Responses and usage

`norm_completion` maps fastllm completions to rishi `Resp`. `UsageStats.cached_tokens` is filled in
when the provider reports prompt caching. `reasoning_tokens` and `cache_creation_tokens` come
through the same way, and are left out when the provider does not report them.

In [ ]:
#| export
def norm_usage(u, model=None):
    "fastllm `Usage` -> a rishi usage dict, including the reasoning and cache-write counts providers bill for."
    if u is None: return {}
    out = {'prompt_tokens': u.prompt_tokens, 'completion_tokens': u.completion_tokens,
           'total_tokens': u.total_tokens or (u.prompt_tokens + u.completion_tokens),
           'cached_tokens': u.cached_tokens, 'model': model}
    for k in ('reasoning_tokens', 'cache_creation_tokens'):
        if (v := getattr(u, k, 0)): out[k] = v
    return out

#: Keys a provider hides a generated image behind, and the mime key beside it.
_inline_keys = (('inlineData', 'mimeType'), ('inline_data', 'mime_type'))

def _d(o):
    "`o` as a plain dict, whether it arrived as one or as a provider's response object."
    if isinstance(o, dict): return o
    for a in ('model_dump', 'dict', 'to_dict'):
        if callable(f := getattr(o, a, None)):
            try: return f()
            except Exception: pass
    return getattr(o, '__dict__', {}) or {}

def _media_from(o):
    "Generated media in one raw response node, as `(mime, bytes)` pairs."
    d, out = _d(o), []
    for k, mk in _inline_keys:
        if (inl := d.get(k)):
            inl = _d(inl)
            if inl.get('data'): out.append((inl.get(mk) or 'image/png', b64decode(inl['data'])))
    u = _d(d.get('image_url') or {}).get('url') or (d.get('image_url') if isinstance(d.get('image_url'), str) else None)
    if u and (du := data_url(u)): out.append((du[0], b64decode(du[1])))
    for k in ('result', 'b64_json'):
        if isinstance(d.get(k), str) and d[k]: out.append(('image/png', b64decode(d[k])))
    return out

def _walk(o, depth=0):
    "Every dict-ish node in a raw response."
    if depth > 6: return
    if isinstance(o, (list, tuple)):
        for x in o: yield from _walk(x, depth+1)
        return
    d = _d(o)
    if not d: return
    yield d
    for v in d.values():
        if isinstance(v, (dict, list, tuple)) or hasattr(v, '__dict__'): yield from _walk(v, depth+1)

def gen_media(raw):
    "Images a model generated, dug out of `Completion.raw`. fastllm's `PartType` is input-only."
    if raw is None: return []
    seen, out = set(), []
    for node in _walk(raw):
        for mime, data in _media_from(node):
            if (h := hash(data)) not in seen:
                seen.add(h); out.append({'mime': mime, 'data': data})
    return out

def norm_completion(comp):
    "fastllm `Completion` -> rishi `Resp`, reading `<tool_call>` tags out of the text as `core.norm_resp` does."
    res = to_hist(comp.message)[0]
    res.setdefault('role', 'assistant')
    tag_tcs = []
    if isinstance(res.get('content'), str): res['content'], tag_tcs = parse_tool_tags(res['content'])
    tcs = [ToolCall(name=tc.name, arguments=tc.arguments, id=tc.id, server=tc.server) for tc in (comp.tool_calls or [])]
    tcs += [ToolCall(name=tc['function']['name'], arguments=tc['function']['arguments'], id=tc['id']) for tc in tag_tcs]
    if tcs: res['tool_calls'] = tcs
    if comp.finish_reason == 'length': res['truncated'] = True
    res['usage'] = norm_usage(comp.usage, comp.model)
    # off `content`, so everything that walks response text is unchanged
    if (media := gen_media(getattr(comp, 'raw', None))): res['media'] = media
    return Resp(res)

In [ ]:
from base64 import b64encode

png = b'\x89PNG\r\n\x1a\n' + b'fake'
b64 = b64encode(png).decode()

# Gemini native, camelCase and snake_case, however deep the candidate sits
test_eq(gen_media({'candidates': [{'content': {'parts': [
    {'text': 'here you go'}, {'inlineData': {'mimeType': 'image/png', 'data': b64}}]}}]}),
    [{'mime': 'image/png', 'data': png}])
test_eq(gen_media({'candidates': [{'content': {'parts': [
    {'inline_data': {'mime_type': 'image/webp', 'data': b64}}]}}]}),
    [{'mime': 'image/webp', 'data': png}])

# the OpenAI-shaped route litellm normalises Gemini's image output onto
test_eq(gen_media({'choices': [{'message': {'content': 'done',
    'images': [{'image_url': {'url': f'data:image/png;base64,{b64}'}}]}}]}),
    [{'mime': 'image/png', 'data': png}])

# the Responses API image tool, and the Images API
test_eq(gen_media({'output': [{'type': 'image_generation_call', 'result': b64}]}),
        [{'mime': 'image/png', 'data': png}])
test_eq(gen_media({'data': [{'b64_json': b64}]}), [{'mime': 'image/png', 'data': png}])

# a plain text reply carries nothing, and neither does a missing `raw`
test_eq(gen_media({'choices': [{'message': {'content': 'just words'}}]}), [])
test_eq(gen_media(None), [])

# the same image reached by two routes in one payload is still one image
test_eq(len(gen_media({'candidates': [{'content': {'parts': [{'inlineData': {'mimeType': 'image/png', 'data': b64}}]}}],
                       'choices': [{'message': {'images': [{'image_url': {'url': f'data:image/png;base64,{b64}'}}]}}]})), 1)

# and it rides on `Resp.media`, leaving `content` alone
r = norm_completion(Completion(model='m', usage=Usage(prompt_tokens=1, completion_tokens=2),
        message=Msg(role='assistant', content=[Text(text='here')]),
        raw={'candidates': [{'content': {'parts': [{'inlineData': {'mimeType': 'image/png', 'data': b64}}]}}]}))
test_eq(r['content'], 'here')
test_eq(r['media'], [{'mime': 'image/png', 'data': png}])

# a caller may hand in anything Completion-shaped; `raw` is newer than the function and
# must not become a requirement -- ramabana's routing test builds one without it
class _NoRaw:
    model, usage, finish_reason, tool_calls = 'm', None, 'stop', None
    message = Msg(role='assistant', content=[Text(text='hi')])
test_eq(norm_completion(_NoRaw())['content'], 'hi')
assert 'media' not in norm_completion(_NoRaw())

## RemoteChat

Async wire via `run_coro` / `sync_iter`. Hosted-only passthrough: `tool_choice`, `reasoning_effort`, and `tool_mode='tags'|'native'`. Provider-run tools return `server=True` and are recorded, not executed locally.


In [ ]:
#| export
#: What `think=False` asks a hosted model for. A name, not a literal, so a deployment whose
#: provider spells the lowest reasoning effort differently can say so.
NO_THINK_EFFORT = 'minimal'
dflt_remote = 'gpt-5.1'      #: model for a `RemoteChat` with none named
DFLT_MAX_TOKENS = 4096

class RemoteChat(ToolLoopMixin, Chat):
    "Chat against a hosted model through fastllm, with the same `Chat` API as the local backends."
    _runtime = 'remote'
    _dflt_cbs = [UsageCallback, ToolReminderCallback, SlidingWindowCallback]
    #: what this transport calls each portable option
    _opt_map = {'ctx': 'ctx_limit', 'temp': 'temperature', 'effort': 'reasoning_effort',
                'max_output_tokens': 'max_tokens'}
    _opt_skip = ('top_k', 'top_p', 'seed', 'think')   # no hosted API takes these
    mk_content, mk_msg, mk_msgs = staticmethod(mk_content), staticmethod(mk_msg), staticmethod(mk_msgs)

    @staticmethod
    def fmt2hist(msgs):
        "fastllm `Msg`s (or canonical dicts) -> canonical history dicts."
        out = []
        for m in listify(msgs): out += to_hist(m) if isinstance(m, Msg) else [mk_msg(m)]
        return out

    @staticmethod
    def hist2fmt(msgs):
        "Canonical history dicts -> fastllm `Msg`s, media carried through rather than stripped."
        return [to_msg(m) for m in listify(msgs) if m.get('role') != 'system']

    def __init__(self, model=None, *, runtime=None, model_path=None, opts=None,
                 vendor_name=None,   # override the vendor fastllm infers from the model id
                 api_name=None,      # ...and the api name
                 tool_choice=None,   # force a particular tool, or 'required'
                 retries=2,
                 comp_kw=None,       # passed to `acomplete` verbatim
                 **kw):              # portable options; see `urai.ChatOpts`
        o = ChatOpts.create(opts, **kw)
        self.model_id = core.split_runtime(model)[1] or dflt_remote
        self.tool_mode = o.tool_mode or 'native'
        self.api_key, self.base_url = o.key, o.base_url or None
        self.reasoning_effort, self.temp = o.effort or None, o.temp
        self.max_output_tokens = o.max_output_tokens or DFLT_MAX_TOKENS
        self.ctx_limit = o.ctx or None
        store_attr('vendor_name,api_name,tool_choice,retries', self)
        self.comp_kw, self._ctx_tokens = comp_kw or {}, 0
        self._set_tools(o.tools)
        self._setup(model, o)

    @property
    def token_count(self):
        "Tokens the last turn reported. Hosted APIs have no live context read-out."
        return self._ctx_tokens

    def _kw(self, stream=False, **turn):
        "Keyword arguments for `acomplete`, with this turn's own generation options applied."
        t = self.map_opts(turn)
        kw = dict(stream=stream, api_key=self.api_key, base_url=self.base_url, retries=self.retries,
                  vendor_name=self.vendor_name, api_name=self.api_name,
                  max_tokens=t.pop('max_tokens', None) or self.max_output_tokens)
        tags = t.pop('tool_mode', self.tool_mode) == 'tags'
        sp = tag_tools_sp(self.toolspecs, self.sp) if tags else self.sp
        if sp: kw['system'] = sp
        # in tag mode the schemas already went out in the system prompt, and sending them on the
        # wire as well is the one thing the transport cannot do
        if self.toolspecs and not tags: kw['tools'] = self.toolspecs
        if self.tool_choice is not None and not tags: kw['tool_choice'] = self.tool_choice
        if (eff := t.pop('reasoning_effort', self.reasoning_effort)) is not None: kw['reasoning_effort'] = eff
        if (tmp := t.pop('temperature', self.temp)) is not None: kw['temperature'] = tmp
        t.pop('ctx_limit', None)                       # a window is not a completion argument
        return {**kw, **self.comp_kw, **t}

    async def _acomplete(self, msgs, stream=False, **turn):
        "One `acomplete` call for `msgs` (canonical history dicts)."
        return await acomplete(self.hist2fmt(msgs), self.model_id, **self._kw(stream, **turn))

    def _model_step(self, **kw):
        "One completion, normalized to a `Resp`. The wire call `ToolLoopMixin` drives."
        return norm_completion(run_coro(self._acomplete(self.hist, False, **kw)))

    def _stream_step(self, **kw):
        "Stream one completion, yielding chunk dicts and leaving the merged `Resp` on `self._step_res`."
        async def _agen():
            agen = await self._acomplete(self.hist, True, **kw)
            async for o in agen: yield o
        comp, split = None, StreamSplit() if self.tool_mode == 'tags' else None
        for o in sync_iter(_agen, stop=self._cancel):
            if isinstance(o, Completion): comp = o
            elif not isinstance(o, Part): continue          # a `Status` marker, not model content
            elif o.type == PartType.tool_use:
                yield {'content': [{'type': 'tool_call', 'name': o.name or '',
                                    'arguments': o.arguments or {}}]}
            elif o.type == PartType.thinking: yield {'channels': {'thought': o.text or ''}}
            elif o.type in (PartType.text, PartType.refusal) and (t := o.text):
                # in tag mode the calls arrive as text, so split it on the way past: otherwise the
                # `<tool_call>` block renders as prose before the final `Resp` turns it into a call
                yield from (split.feed(t) if split else [{'content': [{'type': 'text', 'text': t}]}])
        if split is not None: yield from split.finish()
        if comp is None: raise RuntimeError('stream ended without a final Completion')
        self._step_res = norm_completion(comp)

    def _oneshot(self, prompt, sp='', think=None, max_tokens=None):
        "Stateless completion text, with no history and no tools. `think=False` asks for `NO_THINK_EFFORT`."
        msgs = [to_msg({'role': 'user', 'content': prompt})]
        kw = {**self._kw(max_output_tokens=max_tokens), 'system': sp or None,
              'tools': None, 'tool_choice': None}
        async def _send(k):
            comp = None
            agen = await acomplete(msgs, self.model_id, **{**k, 'stream': True})
            async for part in agen:
                if isinstance(part, Completion): comp = part
            if comp is None: raise RuntimeError('stream ended without a final Completion')
            return comp
        send = lambda k: resp_text(norm_completion(run_coro(_send(k))))
        if think is not False: return send(kw)
        try: return send({**kw, 'reasoning_effort': NO_THINK_EFFORT})
        except Exception: return send(kw)   # a provider that spells the effort differently

    def _structured_call(self, prompt, schema, sp):
        "Force the tool call for `schema` and return its arguments, falling back to JSON found in prose."
        spec = mk_toolspec(schema)
        name = spec['function']['name']
        kw = {**self._kw(), 'system': sp or None, 'tools': [spec], 'tool_choice': name}
        comp = run_coro(acomplete([to_msg({'role': 'user', 'content': prompt})], self.model_id, **kw))
        r = norm_completion(comp)
        if tcs := r.get('tool_calls'): return tcs[0].arguments
        txt = resp_text(r)
        try: return json.loads(extract_fence(txt, 'json'))
        except (json.JSONDecodeError, TypeError):
            raise ValueError(f'model neither called the tool nor returned JSON; reply: {txt[:200]!r}')

    def close(self):
        "Nothing to release. The HTTP client is fastllm's, and cached across chats."
        pass


In [ ]:
def _add(a: int, b: int) -> int:
    'Add a and b.'
    return a + b

# native puts the schemas on the wire; tags puts them in the system prompt and leaves the field empty
kw = RemoteChat('gpt-5.1', tools=[_add], sp='Be terse.')._kw()
test_eq([t['function']['name'] for t in kw['tools']], ['_add'])
test_eq(kw['system'], 'Be terse.')

kw = RemoteChat('gpt-5.1', tools=[_add], sp='Be terse.', tool_mode='tags', tool_choice='required')._kw()
assert 'tools' not in kw and 'tool_choice' not in kw
assert kw['system'].startswith('Be terse.') and '"name": "_add"' in kw['system']

# ...and a tag call in the reply text comes back as a real tool call, with the prose left behind
def _comp(text):
    return Completion(model='m', usage=Usage(prompt_tokens=1, completion_tokens=2),
                      message=Msg(role='assistant', content=[Text(text=text)]))
r = norm_completion(_comp('on it\n<tool_call>{"name": "_add", "arguments": {"a": 1, "b": 2}}</tool_call>'))
test_eq(resp_text(r), 'on it')
test_eq((r['tool_calls'][0].name, r['tool_calls'][0].arguments), ('_add', {'a': 1, 'b': 2}))
test_eq(norm_completion(_comp('just prose')).get('tool_calls'), None)

# the reasoning and cache-write counts providers bill for, carried through when reported and left
# out when they are not, so a local backend's usage dict stays as it was
u = Usage(prompt_tokens=10, completion_tokens=5, cached_tokens=4, reasoning_tokens=7, cache_creation_tokens=3)
test_eq(norm_usage(u, 'm'), {'prompt_tokens': 10, 'completion_tokens': 5, 'total_tokens': 15,
                             'cached_tokens': 4, 'model': 'm', 'reasoning_tokens': 7, 'cache_creation_tokens': 3})
test_eq(set(norm_usage(Usage(prompt_tokens=1, completion_tokens=1), 'm')),
        {'prompt_tokens', 'completion_tokens', 'total_tokens', 'cached_tokens', 'model'})

us = UsageStats(**norm_usage(u)) + UsageStats(**norm_usage(u))
test_eq((us.reasoning_tokens, us.cache_creation_tokens, us.cached_tokens), (14, 6, 8))
assert 'reasoning=14' in repr(us) and 'cache_write=6' in repr(us)
test_eq(sum_usage([norm_usage(u), norm_usage(u)])['reasoning_tokens'], 14)
assert 'reasoning_tokens' not in sum_usage([{'prompt_tokens': 1}])   # nothing to say, so nothing said

In [ ]:
#| hide
# The whole turn, offline. fastllm streams typed parts and ends with a `Completion`, so a fake
# `acomplete` shaped that way covers the conversions, the tool loop and streaming without a key.
# It shadows the imported name, which the `eval: false` cells below re-import to undo.
from fastllm.streaming import Status
from aidialog.msg_parts import Thinking, ToolUse, Refusal

seen, _script = [], []

async def acomplete(msgs, model, **kw):
    "Pop one scripted `Completion`, streaming its parts first when asked."
    seen.append({'msgs': msgs, 'model': model, **kw})
    comp = _script.pop(0)
    if not kw.get('stream'): return comp
    async def gen():
        yield Status('processing')          # not model content, and must not reach the caller
        for p in comp.message.content: yield p
        yield comp
    return gen()

def mk_comp(*parts, **kw):
    "A `Completion` carrying `parts`, as a provider would send one."
    return Completion(model='m', usage=Usage(prompt_tokens=3, completion_tokens=2),
                      message=Msg(role='assistant', content=list(parts)), **kw)

In [ ]:
#| hide
_script[:] = [mk_comp(Text(text='pong'))]
chat = RemoteChat('codex/gpt-5.6-sol')

test_eq(chat.oneshot('ping'), 'pong')
test_eq(seen[-1]['stream'], True)

_script[:] = [mk_comp(Text(text='pong'))]
chat = RemoteChat('gpt-5.1', sp='be brief')
test_eq(resp_text(chat('ping')), 'pong')
test_eq(seen[-1]['system'], 'be brief')
test_eq(chat.use.total_tokens, 5)

# the tool loop: a `ToolUse` part comes back, rishi runs it, the result goes out as a tool message
_script[:] = [mk_comp(ToolUse(id='c1', name='_add', arguments={'a': 2, 'b': 3})), mk_comp(Text(text='5'))]
chat = RemoteChat('gpt-5.1', tools=[_add])
test_eq(resp_text(chat('2+3?')), '5')
sent = seen[-1]['msgs']
test_eq(sent[-1].role, 'tool')
test_eq((sent[-1].content[0].name, sent[-1].content[0].text), ('_add', '5'))
test_eq([p.name for p in sent[-2].content if p.type == PartType.tool_use], ['_add'])
test_eq([p.type for p in sent[-2].content], [PartType.tool_use])   # and no empty text part beside it

# streaming: thinking on its own channel, text as markdown, and one merged `Resp` on `hist`
_script[:] = [mk_comp(Thinking(text='hmm'), Text(text='hello'))]
chat = RemoteChat('gpt-5.1')
out = ''.join(chat('hi', stream=True))
assert 'hello' in out and 'hmm' in out
test_eq((resp_text(chat.hist[-1]), thought(chat.hist[-1])), ('hello', 'hmm'))

_script[:] = [mk_comp(ToolUse(id='c1', name='_add', arguments={'a': 2, 'b': 3})), mk_comp(Text(text='5'))]
chat = RemoteChat('gpt-5.1', tools=[_add])
''.join(chat('2+3?', stream=True))
test_eq(resp_text(chat.hist[-1]), '5')

# a refusal is the reply, so it reaches the caller both ways rather than vanishing
_script[:] = [mk_comp(Refusal(text='I cannot help with that'))]
chat = RemoteChat('gpt-5.1')
assert 'cannot help' in ''.join(chat('...', stream=True))
test_eq(resp_text(chat.hist[-1]), 'I cannot help with that')

## Against a real API

`Chat('gpt-5.1')`, `Chat('anthropic/claude-sonnet-4-5')` and `runtime='remote'` all route here. You
need a vendor key.

In [ ]:
#| eval: false
from fastllm.acomplete import acomplete               # undo the test patch above

chat = Chat('gpt-4.1', sp='You are concise.')
test_eq(chat.runtime, 'remote')
r = chat('Reply with exactly: pong')
assert 'pong' in resp_text(r).lower()
print(chat.use)
chat.oneshot('what do you say to ping in one word')

total=22|in=20|out=2|turns=1|model=gpt-4.1


'The phrase "**say to ping**" can be interpreted in different contexts, but commonly, when someone says they want to "ping" someone (or something), it usually refers to one of the following:\n\n### 1. **In Networking or IT:**\nTo **ping** is to send a small data packet to a server or device to check if it\'s reachable/alive (like `ping google.com` in the command line).\n- **What you say:**  \n  You typically "say" or type the command:  \n  ```\n  ping [IP address or domain]\n  ```\n  For example: `ping google.com`\n\n### 2. **In Chat/Business Communication:**\nTo "ping" someone means to send them a quick message, usually to get their attention or check in.\n- **What you say:**  \n  You might say:  \n  - "I\'ll ping you later."\n  - "Can you ping John to see if he\'s available?"\n  - "Just pinging you to follow up on this."\n\n  Or, if you’re the person "pinging," you might send a message like:\n  - "Hi, just pinging to check in."\n  - "Ping!"\n\n### 3. **If Someone Says “What do you sa

In [ ]:
#| eval: false
display_stream(Chat('gpt-5.1')('Write two sentences about the monsoon.', stream=True))

The monsoon is a seasonal wind system that brings heavy rainfall to regions such as South Asia, Southeast Asia, and parts of Africa. It is crucial for agriculture and water resources but can also cause severe flooding and landslides.

'The monsoon is a seasonal wind system that brings heavy rainfall to regions such as South Asia, Southeast Asia, and parts of Africa. It is crucial for agriculture and water resources but can also cause severe flooding and landslides.'

In [ ]:
#| eval: false
th = Chat('gpt-5.1', reasoning_effort='high', max_output_tokens=2048)
r = th('A bat and ball cost $1.10, the bat is $1 more than the ball. How much is the ball?')
print(thought(r)[:400]); print('---'); print(resp_text(r))


---
Let the price of the ball be \(x\) dollars.  
Then the bat costs \(x + 1\) dollars.

Total cost:
\[
x + (x + 1) = 1.10
\]
\[
2x + 1 = 1.10
\]
\[
2x = 0.10
\]
\[
x = 0.05
\]

The ball costs **5 cents**.


In [ ]:
#| eval: false
img = Path('images.jpeg').read_bytes()
print(resp_text(Chat('gpt-5.1')([img, 'What is in this image? One sentence.'])))

A German Shepherd dog is standing on an outdoor path with its tongue out.


In [ ]:
#| eval: false
# the whole point: start local, finish hosted, with one history
from rishi.llama import LlamaChat, qwen3_17b

local = LlamaChat(qwen3_17b, n_ctx=4096)
local('My name is Karthik and my favourite number is 17. Remember both.')
hist = local.hist
local.close()

remote = Chat('gpt-5.1', messages=hist)
r = remote('What is my name and my favourite number?')
assert 'karthik' in resp_text(r).lower() and '17' in resp_text(r)
resp_text(r)

/Users/71293/code/personal/orgs/ramabana/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
llama_context: n_ctx_seq (4096) < n_ctx_train (40960) -- the full capacity of the model will not be utilized


'Your name is Karthik, and your favourite number is 17.'

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

In [ ]:
empty_parts = _parts([{'type': 'text', 'text': ''}])
test_eq(empty_parts, [])

mixed_parts = _parts([{'type': 'text', 'text': ''}, {'type': 'text', 'text': 'keep'}])
test_eq([p.text for p in mixed_parts], ['keep'])
test_eq(to_msg({'role': 'user', 'content': [{'type': 'text', 'text': ''}]}).content, [])